# Активы для статьи: zero-shot малых моделей + графики

1. **Zero-shot 0.5B/1.5B** на D1 (тест 500, тот же протокол, что у 3B) —
   колонка для таблицы ёмкости. GPU, ~5 мин.
2. **Графики** (CPU, matplotlib, сохранение в `paper/figs/` в PNG+PDF):
   - fig1: dose-response эффекта RL/mask по ёмкости модели;
   - fig2: кривые data-efficiency (0.5B и 3B);
   - fig3: траектории «плохих бассейнов» против здоровых сидов;
   - fig4: динамика награды GRPO.

Порядок: сначала ячейки 1–2 (zero-shot), затем графики. Рисунки
сначала изучить (PNG), в LaTeX пойдут PDF-версии.

In [1]:
# ============================================================
# 1. ZERO-SHOT 0.5B / 1.5B на D1 (тест 500, протокол = ячейка 3B)
# ============================================================
import os, gc, re, json
import torch
import numpy as np
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from datasets import load_dataset
from huggingface_hub import list_repo_files, hf_hub_download

REPO, REV = "empathetic_dialogues", "refs/convert/parquet"
files = list_repo_files(REPO, repo_type="dataset", revision=REV)
names = sorted(f for f in files if f.startswith("default/test/") and f.endswith(".parquet"))
paths = [hf_hub_download(REPO, n, repo_type="dataset", revision=REV) for n in names]
ds = load_dataset("parquet", data_files=paths, split="train")
episodes, seen = [], set()
for r in ds:
    if r["conv_id"] in seen:
        continue
    seen.add(r["conv_id"])
    t, l = r["prompt"].strip(), r["context"].strip()
    if t and l:
        episodes.append((t, l))
episodes = episodes[:500]
print(f"тест-эпизодов: {len(episodes)}")

emotion_labels = sorted({l for _, l in episodes})
def _norm(s):
    return re.sub(r"[^a-z ]", "", s.lower()).strip()
EMOTION_BY_NORM = {_norm(l): l for l in emotion_labels}
def match_emotion(text):
    g = _norm(text)
    if not g:
        return None
    first = g.split()[0]
    if first in EMOTION_BY_NORM:
        return EMOTION_BY_NORM[first]
    if g in EMOTION_BY_NORM:
        return EMOTION_BY_NORM[g]
    for nl, l in EMOTION_BY_NORM.items():
        if g.startswith(nl):
            return l
    for nl, l in EMOTION_BY_NORM.items():
        if nl in g:
            return l
    return None

RESULTS = {"Qwen/Qwen2.5-3B": 0.2580}   # из ячейки d220af6b
for model_name in ("Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-1.5B"):
    set_seed(42)
    tok = AutoTokenizer.from_pretrained(model_name)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(
        model_name, dtype=torch.bfloat16).eval().to("cuda")
    correct, examples = 0, []
    with torch.no_grad():
        for i in tqdm(range(0, len(episodes), 16), desc=model_name.split("/")[-1]):
            batch = episodes[i:i + 16]
            prompts = [f"Situation: {t}\nEmotion:" for t, _ in batch]
            enc = tok(prompts, return_tensors="pt", padding=True,
                      truncation=True, max_length=104).to(m.device)
            out = m.generate(**enc, max_new_tokens=6, do_sample=False,
                             num_beams=1, pad_token_id=tok.pad_token_id,
                             eos_token_id=tok.eos_token_id)
            for j in range(enc["input_ids"].size(0)):
                pred = tok.decode(out[j, enc["input_ids"].size(1):],
                                  skip_special_tokens=True).strip()
                correct += int(match_emotion(pred) == batch[j][1])
                if len(examples) < 4:
                    examples.append((batch[j][1], match_emotion(pred), pred))
    RESULTS[model_name] = correct / len(episodes)
    print(f"{model_name}: zero-shot accuracy = {RESULTS[model_name]:.4f}")
    for l, p, raw in examples:
        print(f"  {l:<14} -> {str(p):<14} | {raw[:40]!r}")
    del m
    gc.collect(); torch.cuda.empty_cache()

os.makedirs("paper/data", exist_ok=True)
with open("paper/data/zeroshot_models.json", "w") as f:
    json.dump(RESULTS, f, indent=2)
print("\nСохранено: paper/data/zeroshot_models.json")

<VENV>/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


тест-эпизодов: 500


Qwen2.5-0.5B: 100%|██████████| 32/32 [00:04<00:00,  7.25it/s]


Qwen/Qwen2.5-0.5B: zero-shot accuracy = 0.1560
  guilty         -> angry          | 'I was angry and hurt.'
  caring         -> None           | 'in need. We took them'
  lonely         -> None           | 'I feel so empty.\nI'
  excited        -> None           | 'I am happy.\nQuestion:'


Qwen2.5-1.5B: 100%|██████████| 32/32 [00:05<00:00,  5.44it/s]


Qwen/Qwen2.5-1.5B: zero-shot accuracy = 0.1840
  guilty         -> guilty         | 'Guilty\nWhat is the'
  caring         -> None           | 'in need of help. I'
  lonely         -> sad            | 'Sadness\nWhat is the'
  excited        -> excited        | 'Excited\nI want to'

Сохранено: paper/data/zeroshot_models.json


In [2]:
# ============================================================
# 2. Общие настройки графики
# ============================================================
import json, os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300, "font.size": 10,
    "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True,
    "font.family": "DejaVu Sans",
})
FIGS = "paper/figs"
os.makedirs(FIGS, exist_ok=True)
C = {"MLE": "#555555", "mask": "#1f77b4", "RL": "#d62728"}

def save(fig, name):
    fig.savefig(f"{FIGS}/{name}.png", bbox_inches="tight")
    fig.savefig(f"{FIGS}/{name}.pdf", bbox_inches="tight")
    print(f"сохранено: {FIGS}/{name}.png (+.pdf)")

ld = json.load(open("sweep_ld/summary_lowdata.json"))["rows"]
def cell(m, size, cfg):
    rs = [r for r in ld if r["model"] == m and r["size"] == size]
    vals = [r[cfg] for r in rs]
    return np.mean(vals), np.std(vals), len(rs)
print("данные загружены; ячеек:", len(ld))

данные загружены; ячеек: 38


In [4]:
# ============================================================
# 3. РИС. 1 — dose-response эффекта по ёмкости модели (n=6000)
# ============================================================
models = [("0.5B", "05B"), ("1.5B", "15B"), ("3B", "3B")]
zeroshot = json.load(open("paper/data/zeroshot_models.json")) if \
    os.path.exists("paper/data/zeroshot_models.json") else {}

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, cfg, title in zip(axes, ("RL", "mask"),
                          ("RL $-$ MLE", "Контраст (маска) $-$ MLE")):
    xs, ys, es, ns = [], [], [], []
    for i, (label, m) in enumerate(models):
        mean, std, n = cell(m, 6000, cfg)
        mle_mean = cell(m, 6000, "MLE")[0]
        # парная разность: пересчитаем честно по сидам
        rs = [r for r in ld if r["model"] == m and r["size"] == 6000]
        d = [r[cfg] - r["MLE"] for r in rs]
        xs.append(i); ys.append(np.mean(d))
        es.append(np.std(d, ddof=1) / np.sqrt(len(d)) if len(d) > 1 else 0)
        ns.append(len(d))
    ax.errorbar(xs, ys, yerr=es, marker="o", markersize=7,
                color=C[cfg], capsize=4, linewidth=1.5)
    ax.axhline(0, color="k", linewidth=0.8, linestyle="--")
    # звёздочка значимости снята: после репликации RL-эффект 0.5B
    # t=1.9 < 2.262 (n=10) — отмечать значимость нельзя (§10.3 журнала)
    ax.set_xticks(xs)
    ax.set_xticklabels([f"{l}\n(n={n})" for (l, _), n in zip(models, ns)])
    ax.set_ylabel("Парная разность точности")
    ax.set_title(title)
    ax.set_ylim(-0.014, 0.019)
fig.suptitle("Зависимость эффекта от ёмкости модели "
             "(EmpatheticDialogues, 6000 примеров, единый протокол)", y=1.04)
fig.tight_layout()
save(fig, "fig1_capacity")

# текстом: уровни MLE и zero-shot для подписи в статье
for label, m in models:
    mm, ms, mn = cell(m, 6000, "MLE")
    print(f"{label}: MLE {mm:.3f}±{ms:.3f} (n={mn})")
print("zero-shot:", {k.split("/")[-1]: round(v, 3) for k, v in zeroshot.items()})

сохранено: paper/figs/fig1_capacity.png (+.pdf)
0.5B: MLE 0.541±0.005 (n=10)
1.5B: MLE 0.584±0.007 (n=3)
3B: MLE 0.620±0.006 (n=3)
zero-shot: {'Qwen2.5-3B': 0.258, 'Qwen2.5-0.5B': 0.156, 'Qwen2.5-1.5B': 0.184}


In [ ]:
# ============================================================
# 4. РИС. 2 — кривые data-efficiency (0.5B и 3B)
# ============================================================
from matplotlib.ticker import NullFormatter

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6), sharey=False)
for ax, (label, m, sizes) in zip(axes, [
        ("Qwen2.5-0.5B", "05B", [250, 1000, 6000]),
        ("Qwen2.5-3B", "3B", [250, 500, 1000, 6000])]):
    for cfg in ("MLE", "mask", "RL"):
        xs, ys, es = [], [], []
        for s in sizes:
            rs = [r for r in ld if r["model"] == m and r["size"] == s]
            if not rs:
                continue
            vals = [r[cfg] for r in rs]
            xs.append(s); ys.append(np.mean(vals))
            es.append(np.std(vals, ddof=1) / np.sqrt(len(vals))
                      if len(vals) > 1 else 0)
        ax.errorbar(xs, ys, yerr=es, marker="o", markersize=5,
                    color=C[cfg], label={"MLE": "MLE",
                    "mask": "маска", "RL": "RL"}[cfg], capsize=3)
    zs = zeroshot.get(f"Qwen/Qwen2.5-{label.split('-')[-1]}")
    if zs is None:
        zs = {"Qwen2.5-0.5B": None, "Qwen2.5-3B": 0.258}.get(label, None)
    if zs is not None:
        ax.axhline(zs, color="#7f7f7f", linestyle=":", linewidth=1.2)
        ax.text(0.98, zs + 0.004, "zero-shot", ha="right", fontsize=8,
                color="#7f7f7f", transform=ax.get_yaxis_transform())
    ax.set_xscale("log")
    ax.set_xticks(sizes)
    ax.set_xticklabels([str(s) for s in sizes])
    ax.tick_params(axis="x", which="minor", labelbottom=False)  # ← вот это
    ax.set_xlabel("Обучающих примеров")
    ax.set_title(label)
axes[0].set_ylabel("Точность классификации")
axes[0].legend(loc="lower right", framealpha=0.9)
fig.suptitle("Точность по объёму обучающих данных "
             "(сглаживание: 3–10 сидов на точку)", y=1.03)
fig.tight_layout()
save(fig, "fig2_dataeff")

сохранено: paper/figs/fig2_dataeff.png (+.pdf)


In [5]:
# ============================================================
# 5. РИС. 3 — «плохие бассейны»: траектории val
# ============================================================
def hist(path):
    """История val; если основная папка была перезаписана прерванным
    прогоном (короткая) — берём полную копию из каузального повтора
    (значения совпадают бит-в-бит по построению теста)."""
    with open(path) as f:
        h = [x["value"] for x in json.load(f)["history"]]
    if len(h) < 4 and "d21_seed42" in path and "incoherent" in path:
        alt = path.replace("d21_seed42", "d23_seed42_incoherent-repro")
        with open(alt) as f:
            h = [x["value"] for x in json.load(f)["history"]]
    return h

cases = [
    ("Генерация, переставленный негатив (сид 42)",
     "gen_d2_incoherent_d21_seed42/val_metric_history.json",
     "gen_d2_mle_d21_seed42/val_metric_history.json", "rougeL"),
    ("Генерация, маска промпта (сид 48)",
     "gen_d2_mask_d21_seed48/val_metric_history.json",
     "gen_d2_mle_d21_seed48/val_metric_history.json", "rougeL"),
    ("Классификация, путаемая метка (сид 48)",
     "gen_dialogue_ed_contrast_wl_d16_seed48/val_metric_history.json",
     "gen_crl_mle_baseline_d16_seed48/val_metric_history.json", "accuracy"),
]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.3))
for ax, (title, bad_p, mle_p, metric) in zip(axes, cases):
    bad, good = hist(bad_p), hist(mle_p)
    ax.plot(range(1, len(good) + 1), good, marker="o",
            color=C["MLE"], label="MLE (тот же старт)")
    ax.plot(range(1, len(bad) + 1), bad, marker="s",
            color="#ff7f0e", label="контраст")
    ax.set_xlabel("Эпоха")
    ax.set_title(title, fontsize=9)
axes[0].set_ylabel("ROUGE-L (val)")
axes[2].set_ylabel("Точность (val)")
axes[0].legend(fontsize=8)
fig.suptitle("Воспроизводимые деградированные траектории при общих стартах "
             "(единая пара запусков на панель)", y=1.04)
fig.tight_layout()
save(fig, "fig3_basins")

сохранено: paper/figs/fig3_basins.png (+.pdf)


In [6]:
# ============================================================
# 6. РИС. 4 — GRPO: динамика награды и разброса групп
# ============================================================
fig, ax = plt.subplots(figsize=(6.5, 3.5))
for seed, color in zip((42, 43, 44), ("#d62728", "#1f77b4", "#2ca02c")):
    with open(f"gen_d2_grpo_d24_seed{seed}/results.json") as f:
        h = json.load(f)["rl_reward_history"]
    steps = [x["step"] for x in h]
    rew = [x["reward"] for x in h]
    ax.plot(steps, rew, marker="o", markersize=3.5, color=color,
            linewidth=1.3, label=f"сид {seed}")
    # скользящее среднее для читаемости
    if len(rew) >= 5:
        k = 5
        sm = np.convolve(rew, np.ones(k) / k, mode="valid")
        ax.plot(steps[k - 1:], sm, color=color, linewidth=2.6, alpha=0.45)
ax.set_xlabel("Шаг обучения")
ax.set_ylabel("Награда (ROUGE-L, hold-out)")
ax.set_title("GRPO: награда снижается в ходе обучения при стабильном "
             "разбросе групп")
ax.legend()
fig.tight_layout()
save(fig, "fig4_grpo")

# групповой std отдельным графиком
fig, ax = plt.subplots(figsize=(6.5, 2.6))
for seed, color in zip((42, 43, 44), ("#d62728", "#1f77b4", "#2ca02c")):
    with open(f"gen_d2_grpo_d24_seed{seed}/results.json") as f:
        h = json.load(f)["rl_reward_history"]
    ax.plot([x["step"] for x in h], [x["group_std"] for x in h],
            marker="o", markersize=3.5, color=color, linewidth=1.2,
            label=f"сид {seed}")
ax.set_xlabel("Шаг обучения")
ax.set_ylabel("Std награды в группе")
ax.set_title("Исследование не вырождается: разброс наград внутри групп")
ax.legend()
fig.tight_layout()
save(fig, "fig4b_grpo_groupstd")

сохранено: paper/figs/fig4_grpo.png (+.pdf)
сохранено: paper/figs/fig4b_grpo_groupstd.png (+.pdf)
